# Time Series Activity Forecasting Process

This notebook outlines a detailed process for forecasting activity (e.g., "seen" events) for various indicators over time using a combination of feature engineering, statistical models, machine learning, and ensemble methods. Below is a step-by-step description of the workflow:

---

## 1. Data Loading and Preparation

- **Data Source:** The data is loaded from a CSV file containing time series records for multiple indicators.
- **Preprocessing:**
    - The `date` column is parsed as a datetime object.
    - Data is sorted by `indicator` and `date`.
    - Only the most recent `n_days` (e.g., 100) are retained for analysis.
    - The resulting DataFrame (`df`) contains columns such as `API_UserName`, `date`, `indicator`, `observations`, `dayofweek`, `is_weekend`, `day`, `month`, and `seen`.

---

## 2. Feature Engineering

- **Time Series Features:** For each indicator, the following features are extracted:
    - `last_seen`: Days since the indicator was last seen.
    - `freq_7`, `freq_30`: Number of times seen in the last 7 and 30 days.
    - `avg_gap`: Average gap between seen events.
    - `burstiness`: Variability in the gaps between events.
    - `label_7`, `label_14`, `label_30`: Binary labels indicating if the indicator was seen in the last 7, 14, or 30 days.
- **Output:** A features DataFrame (`features_df`) indexed by indicator.

---

## 3. Model Training and Prediction

- **Models Used:**
    - **Logistic Regression:** Predicts the probability of being seen in the next 7, 14, or 30 days.
    - **Gradient Boosted Trees (GBT):** Another probabilistic classifier for the same task.
    - **Exponential Model:** Uses recent frequency to estimate the probability of being seen.
    - **Weibull AFT Model:** Survival analysis model to estimate the probability of future events.
- **Predictions:** For each indicator, probabilities are computed for being seen today, in 7, 14, and 30 days.

---

## 4. Rule-Based and Ensemble Methods

- **Rule-Based Labels:** Simple heuristics based on `last_seen` (e.g., if last seen within 7 days, label as active).
- **Logistic Model on Rules:** Trains a logistic regression model using the rule-based labels.
- **Ensemble:** Combines probabilities from the logistic model, GBT, Weibull, and exponential models using weighted averages to improve robustness.

---

## 5. Confidence Assessment and Formatting

- **Confidence Labels:** For each forecast window (today, 7d, 14d, 30d), assigns a qualitative confidence label (e.g., "Highly likely", "Possibly active", "Low confidence") based on probability thresholds and recent frequency.
- **Formatting:** Probabilities are formatted as percentages for presentation.

---

## Summary

This process enables robust, interpretable, and actionable forecasting of indicator activity using a blend of statistical and machine learning techniques, with clear outputs for operational use.

In [1]:
import numpy as np
import pandas as pd
import os
import datetime
from datetime import datetime
from datetime import timedelta
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from lifelines import WeibullAFTFitter
import warnings

In [2]:
from datetime import datetime, timedelta

# Set start_full_date_str and start_date_str to 100 days before today
today = datetime.today()
start_full_date_str = (today - timedelta(days=100)).strftime("%Y-%m-%d")
start_date_str = start_full_date_str
# Define the file path template
file_path_template = "Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"


In [3]:
# Calculate today's date
today = datetime.today()
print(today)

# Calculate end date (today + 0 days)
end_dt = today + timedelta(days=0)
end_date_str = end_dt.strftime("%Y-%m-%d")
print(end_date_str)

# Convert string dates to datetime objects
start_full_dt = datetime.strptime(start_full_date_str, "%Y-%m-%d")
start_dt = datetime.strptime(start_date_str, "%Y-%m-%d")

2025-06-24 07:30:42.611842
2025-06-24


In [4]:
# Function to load files and save them to the specified directory
def load_files(filenames):
    dataframes = []
    for filename in filenames:
        if not os.path.exists(filename):
            print(f"File {filename} does not exist. Skipping.")
            continue
        df = pd.read_csv(filename)
        dataframes.append(df)
    return dataframes


# Define the file path template and date range
date_format = "%Y%m%d"
start_date = datetime.strptime(start_date_str, "%Y-%m-%d")  # Convert start_date_str to datetime
dates_to_pull = pd.date_range(start_date, end_dt, freq='D')

# Generate the list of file paths
datelist = [file_path_template.format(date=dt.strftime(date_format)) for dt in dates_to_pull]
print(datelist)

# Concatenate dataframes from the list of files
src = pd.concat(load_files(datelist), ignore_index=True)

# Clean up the 'indicator' and 'OpDiv' columns if they exist
if 'indicator' in src.columns:
    src['indicator'] = src['indicator'].astype(str).str.split(' ', expand=True)[0].str.strip()
if 'OpDiv' in src.columns:
    src['OpDiv'] = src['OpDiv'].astype(str).str.strip()

# Display the cleaned dataframe
display(src)

['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250316.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250317.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250318.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250319.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250320.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250321.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250322.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250323.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250324.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250325.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250326.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250327.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/hto

,indicator,API_UserName,obs_date,OpDiv,indicator_key,observations,curr_date
0,102.90.61.13,64853912235916233429,2025-03-16,OS,102.90.61.13_OS,3117,2025-03-16
1,102.91.94.193,64853912235916233429,2025-03-16,OS,102.91.94.193_OS,2195,2025-03-16
2,103.225.136.166,64853912235916233429,2025-03-16,OS,103.225.136.166_OS,4,2025-03-16
3,104.18.68.40,15920309593055310684,2025-03-16,CMS,104.18.68.40_CMS,5,2025-03-16
4,104.18.69.40,15920309593055310684,2025-03-16,CMS,104.18.69.40_CMS,5,2025-03-16
...,...,...,...,...,...,...,...
81418,104.152.52.148,74198686107399967946,2025-06-24,VA,104.152.52.148_VA,1,2025-06-24
81419,159.13.45.83,20790633968691748718,2025-06-24,DHA,159.13.45.83_DHA,1,2025-06-24
81420,45.138.16.240,64853912235916233429,2025-06-24,OS,45.138.16.240_OS,1,2025-06-24
81421,196.251.70.216,64853912235916233429,2025-06-24,OS,196.251.70.216_OS,1,2025-06-24


In [5]:
src.drop(columns=['curr_date', 'indicator_key'], inplace=True)
src.rename(columns={'obs_date': 'date'}, inplace=True)
src['date'] = pd.to_datetime(src['date'])
src.reset_index(drop=True, inplace=True)


In [6]:
src

,indicator,API_UserName,date,OpDiv,observations
0,102.90.61.13,64853912235916233429,2025-03-16,OS,3117
1,102.91.94.193,64853912235916233429,2025-03-16,OS,2195
2,103.225.136.166,64853912235916233429,2025-03-16,OS,4
3,104.18.68.40,15920309593055310684,2025-03-16,CMS,5
4,104.18.69.40,15920309593055310684,2025-03-16,CMS,5
...,...,...,...,...,...
81418,104.152.52.148,74198686107399967946,2025-06-24,VA,1
81419,159.13.45.83,20790633968691748718,2025-06-24,DHA,1
81420,45.138.16.240,64853912235916233429,2025-06-24,OS,1
81421,196.251.70.216,64853912235916233429,2025-06-24,OS,1


In [7]:
src[src['indicator'] == '192.124.249.112']

,indicator,API_UserName,date,OpDiv,observations
11221,192.124.249.112,50189120947314147395,2025-04-15,NIH,4
11910,192.124.249.112,80363974983666420473,2025-04-16,IHS,13
11911,192.124.249.112,50189120947314147395,2025-04-16,NIH,151
11912,192.124.249.112,74198686107399967946,2025-04-16,VA,14
12649,192.124.249.112,00818860012482918321,2025-04-17,CDC,7
...,...,...,...,...,...
69013,192.124.249.112,74198686107399967946,2025-06-18,VA,29
72802,192.124.249.112,50189120947314147395,2025-06-20,NIH,7
76778,192.124.249.112,50189120947314147395,2025-06-22,NIH,3
78620,192.124.249.112,00818860012482918321,2025-06-23,CDC,44


In [8]:
# Group the src dataframe by 'OpDiv' and get a dictionary of DataFrames for each OpDiv
opdiv_groups = {opdiv: group for opdiv, group in src.groupby('OpDiv')}

# Allow searching by indicator
def get_by_indicator(indicator_value):
    return src[src['indicator'] == indicator_value]

# Example usage:
get_by_indicator('192.124.249.112')

,indicator,API_UserName,date,OpDiv,observations
11221,192.124.249.112,50189120947314147395,2025-04-15,NIH,4
11910,192.124.249.112,80363974983666420473,2025-04-16,IHS,13
11911,192.124.249.112,50189120947314147395,2025-04-16,NIH,151
11912,192.124.249.112,74198686107399967946,2025-04-16,VA,14
12649,192.124.249.112,00818860012482918321,2025-04-17,CDC,7
...,...,...,...,...,...
69013,192.124.249.112,74198686107399967946,2025-06-18,VA,29
72802,192.124.249.112,50189120947314147395,2025-06-20,NIH,7
76778,192.124.249.112,50189120947314147395,2025-06-22,NIH,3
78620,192.124.249.112,00818860012482918321,2025-06-23,CDC,44


In [9]:
opdiv_merged = {}

for opdiv, group_df in opdiv_groups.items():
    group_df['date'] = pd.to_datetime(group_df['date'])
    all_users = group_df['API_UserName'].unique()
    all_indicators = group_df['indicator'].unique()
    all_dates = pd.date_range(start=group_df['date'].min(), end=pd.Timestamp.now().normalize(), freq='D')
    all_combinations = pd.MultiIndex.from_product(
        [all_users, all_dates, all_indicators],
        names=['API_UserName', 'date', 'indicator']
    ).to_frame(index=False)
    all_combinations['OpDiv'] = opdiv  # Add OpDiv column

    merged = all_combinations.merge(group_df, how='left', on=['API_UserName', 'date', 'indicator', 'OpDiv'])
    merged['observations'] = merged['observations'].fillna(0).astype(int)
    merged['date'] = pd.to_datetime(merged['date'])
    merged['dayofweek'] = merged['date'].dt.dayofweek
    merged['is_weekend'] = merged['dayofweek'].isin([5, 6])
    merged['day'] = merged['date'].dt.day
    merged['month'] = merged['date'].dt.month
    merged['seen'] = (merged['observations'] > 0).astype(int)
    opdiv_merged[opdiv] = merged

# Example: display the merged dataframe for one OpDiv
display(opdiv_merged['DHA'])


,API_UserName,date,indicator,OpDiv,observations,dayofweek,is_weekend,day,month,seen
0,20790633968691748718,2025-03-26,104.160.6.2,DHA,2,2,False,26,3,1
1,20790633968691748718,2025-03-26,141.98.252.143,DHA,31,2,False,26,3,1
2,20790633968691748718,2025-03-26,146.71.50.198,DHA,4587,2,False,26,3,1
3,20790633968691748718,2025-03-26,191.96.36.115,DHA,3,2,False,26,3,1
4,20790633968691748718,2025-03-26,193.32.162.136,DHA,3,2,False,26,3,1
...,...,...,...,...,...,...,...,...,...,...
73341,20790633968691748718,2025-06-24,193.163.125.95,DHA,4,1,False,24,6,1
73342,20790633968691748718,2025-06-24,42.247.1.188,DHA,0,1,False,24,6,0
73343,20790633968691748718,2025-06-24,89.248.163.57,DHA,0,1,False,24,6,0
73344,20790633968691748718,2025-06-24,91.191.209.18,DHA,63,1,False,24,6,1


In [10]:
# Replace 'VA' with the correct OpDiv if needed
opdiv_merged['VA'][opdiv_merged['VA']['indicator'] == '192.124.249.112']

,API_UserName,date,indicator,OpDiv,observations,dayofweek,is_weekend,day,month,seen
144,74198686107399967946,2025-03-21,192.124.249.112,VA,0,4,False,21,3,0
1026,74198686107399967946,2025-03-22,192.124.249.112,VA,0,5,True,22,3,0
1908,74198686107399967946,2025-03-23,192.124.249.112,VA,0,6,True,23,3,0
2790,74198686107399967946,2025-03-24,192.124.249.112,VA,0,0,False,24,3,0
3672,74198686107399967946,2025-03-25,192.124.249.112,VA,0,1,False,25,3,0
...,...,...,...,...,...,...,...,...,...,...
80406,74198686107399967946,2025-06-20,192.124.249.112,VA,0,4,False,20,6,0
81288,74198686107399967946,2025-06-21,192.124.249.112,VA,0,5,True,21,6,0
82170,74198686107399967946,2025-06-22,192.124.249.112,VA,0,6,True,22,6,0
83052,74198686107399967946,2025-06-23,192.124.249.112,VA,0,0,False,23,6,0


In [11]:


def load_data(filepath, n_days=100):
    df = pd.read_csv(filepath)
    df['date'] = pd.to_datetime(df['date'])
    df.sort_values(by=['indicator', 'date'], inplace=True)
    latest_dates = df['date'].drop_duplicates().sort_values().tail(n_days)
    return df[df['date'].isin(latest_dates)].copy()

def extract_time_series_features(group):
    series = group['seen'].values
    indices = np.where(series == 1)[0]
    if len(indices) == 0:
        return pd.Series({
            'last_seen': len(series),
            'freq_7': 0,
            'freq_14': 0,
            'freq_30': 0,
            'avg_gap': len(series),
            'burstiness': 0,
            'label_7': 0,
            'label_14': 0,
            'label_30': 0
        })
    last_seen = len(series) - 1 - indices[-1]
    freq_7 = np.sum(series[-7:])
    freq_14 = np.sum(series[-14:])
    freq_30 = np.sum(series[-30:])
    gaps = np.diff(indices)
    avg_gap = np.mean(gaps) if len(gaps) > 0 else len(series)
    burstiness = (np.std(gaps) - avg_gap) / (np.std(gaps) + avg_gap) if len(gaps) > 1 else 0
    label_7 = 1 if np.any(series[-7:]) else 0
    label_14 = 1 if np.any(series[-14:]) else 0
    label_30 = 1 if np.any(series[-30:]) else 0
    return pd.Series({
        'last_seen': last_seen,
        'freq_7': freq_7,
        'freq_14': freq_14,
        'freq_30': freq_30,
        'avg_gap': avg_gap,
        'burstiness': burstiness,
        'label_7': label_7,
        'label_14': label_14,
        'label_30': label_30
    })

def build_features(df):
    features_df = df.groupby('indicator').apply(extract_time_series_features).reset_index()
    return features_df

def train_predict(model_cls, X, y):
    if len(np.unique(y)) < 2:
        return np.full(len(y), np.nan)
    model = Pipeline([('scaler', StandardScaler()), ('clf', model_cls())])
    model.fit(X, y)
    return model.predict_proba(X)[:, 1]

def train_gbt(X, y):
    if len(np.unique(y)) < 2:
        return np.full(len(y), np.nan)
    model = GradientBoostingClassifier()
    model.fit(X, y)
    return model.predict_proba(X)[:, 1]

def fit_weibull_aft(X, avg_gap, event):
    aft_df = X.copy()
    aft_df['duration'] = avg_gap
    aft_df['event'] = event
    aft = WeibullAFTFitter()
    aft.fit(aft_df, duration_col='duration', event_col='event')
    return aft

def get_model_outputs(features_df, df):
    df_pred = features_df.copy()
    X = df_pred[['last_seen', 'freq_7', 'freq_14' ,'freq_30', 'avg_gap', 'burstiness']]
    y_7 = df_pred['label_7']
    y_14 = df_pred['label_14']
    y_30 = df_pred['label_30']

    # Logistic Regression, Estimate baseline probabilities
    df_pred['logistic_7'] = train_predict(LogisticRegression, X, y_7)
    df_pred['logistic_14'] = train_predict(LogisticRegression, X, y_14)
    df_pred['logistic_30'] = train_predict(LogisticRegression, X, y_30)

    # Gradient Boosted Trees, Non-linear patterns and feature interactions
    df_pred['gbt_7'] = train_gbt(X, y_7)
    df_pred['gbt_14'] = train_gbt(X, y_14)
    df_pred['gbt_30'] = train_gbt(X, y_30)

    # Exponential Model, memoryless Poisson process for frequency
    rate = (df_pred['freq_30'] / 30).clip(lower=1e-6)
    df_pred['exp_7'] = 1 - np.exp(-rate * 7)
    df_pred['exp_14'] = 1 - np.exp(-rate * 14)
    df_pred['exp_30'] = 1 - np.exp(-rate * 30)
    df_pred['exp_today'] = 1 - np.exp(-rate * 1)

    # Weibull AFT Model, time-to-event behavor, burstiness
    aft = fit_weibull_aft(X, df_pred['avg_gap'], y_7)
    surv_func = aft.predict_survival_function(X.assign(duration=df_pred['avg_gap'], event=y_7), times=[1, 7, 14, 30])
    df_pred['weibull_today'] = 1 - surv_func.loc[1].values
    df_pred['weibull_7'] = 1 - surv_func.loc[7].values
    df_pred['weibull_14'] = 1 - surv_func.loc[14].values
    df_pred['weibull_30'] = 1 - surv_func.loc[30].values

    # Today's forecast
    df_pred['logistic_today'] = train_predict(LogisticRegression, X, y_7)
    df_pred['gbt_today'] = train_gbt(X, y_7)

    # Merge in actual "seen" value for today's date
    latest_date = df['date'].max()
    today_seen = df[df['date'] == latest_date][['indicator', 'seen']].rename(columns={'seen': 'seen_today'})
    df_pred = df_pred.merge(today_seen, on='indicator', how='left')

    return df_pred

def add_rule_and_ensemble(output):
    features = ['last_seen', 'freq_7', 'freq_30', 'avg_gap', 'burstiness']
    X = output[features]

    # Rule-based labels
    output['rule_today'] = output['last_seen'].apply(lambda x: 1 if x == 0 else 0)
    output['rule_7d'] = output['last_seen'].apply(lambda x: 1 if x <= 6 else 0)
    output['rule_14d'] = output['last_seen'].apply(lambda x: 1 if x <= 13 else 0)
    output['rule_30d'] = output['last_seen'].apply(lambda x: 1 if x <= 29 else 0)

    y_today = output['rule_today']
    y_7 = output['rule_7d']
    y_14 = output['rule_14d']
    y_30 = output['rule_30d']

    def train_logistic_model(X, y):
        if len(np.unique(y)) < 2:
            return np.full(len(y), np.nan)
        model = Pipeline([
            ('scaler', StandardScaler()),
            ('clf', LogisticRegression())
        ])
        model.fit(X, y)
        return model.predict_proba(X)[:, 1]

    output['prob_today'] = train_logistic_model(X, y_today)
    output['prob_7d'] = train_logistic_model(X, y_7)
    output['prob_14d'] = train_logistic_model(X, y_14)
    output['prob_30d'] = train_logistic_model(X, y_30)

    # Ensemble, combines predictions from all models and prevents overfitting
    output['ensemble_7d'] = (
        0.3 * output['prob_7d'].astype(float) +
        0.25 * output['gbt_7'] +
        0.25 * output['weibull_7'] +
        0.3 * output['exp_7']
    )
    output['ensemble_14d'] = (
        0.3 * output['prob_14d'].astype(float) +
        0.25 * output['gbt_14'] +
        0.25 * output['weibull_14'] +
        0.3 * output['exp_14']
    )
    output['ensemble_30d'] = (
        0.3 * output['prob_30d'].astype(float) +
        0.25 * output['gbt_30'] +
        0.25 * output['weibull_30'] +
        0.3 * output['exp_30']
    )
    return output

def classify_window(prob, freq, high_thresh, label):
    if prob >= high_thresh and freq >= 2:
        return f"{label}: Highly likely"
    elif prob >= 0.07 and freq >= 1:
        return f"{label}: Possibly active"
    else:
        return f"{label}: Low confidence"

def add_confidence_and_format(output):
    output['confidence_7d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_7d']), row['freq_7'], 0.6, '7-Day'), axis=1
    )
    output['confidence_14d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_14d']), row['freq_14'], 0.6, '14-Day'), axis=1
    )
    output['confidence_30d'] = output.apply(
        lambda row: classify_window(float(row['ensemble_30d']), row['freq_30'], 0.6, '30-Day'), axis=1
    )

    # Format percentages
    for col in ['prob_7d', 'prob_14d', 'prob_30d', 'ensemble_7d', 'ensemble_14d', 'ensemble_30d']:
        output[col] = np.clip(output[col].astype(float) * 100, 0, 100).round(2).astype(str) + '%'
    return output

def build_production_output(output):
    warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
    production_output = output[[
        'indicator', 'seen_today', 'freq_7', 'freq_30',
        'ensemble_7d', 'confidence_7d',
        'ensemble_14d', 'confidence_14d',
        'ensemble_30d', 'confidence_30d'
    ]].copy()
    production_output.rename(columns={
        'indicator': 'Indicator',
        'seen_today': 'Observed Today',
        'freq_7': 'Frequency (7d)',
        'freq_30': 'Frequency (30d)',
        'ensemble_7d': 'Probability: 7-Day',
        'confidence_7d': 'Confidence: 7-Day',
        'ensemble_14d': 'Probability: 14-Day',
        'confidence_14d': 'Confidence: 14-Day',
        'ensemble_30d': 'Probability: 30-Day',
        'confidence_30d': 'Confidence: 30-Day'
    }, inplace=True)
    return production_output


In [16]:
import pandas as pd
import os
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

def update_long_forecast_log_with_formatting(production_df, actuals_df, log_csv_path=None, excel_output_path=None):
    today = actuals_df['date'].max().normalize()
    records = []

    for h in [7, 14, 30]:
        tag_col = f'Confidence: {h}-Day'
        target_date = today + pd.Timedelta(days=h)

        actuals_on_target = actuals_df[actuals_df['date'] == target_date][['indicator', 'seen']]
        actuals_map = dict(zip(actuals_on_target['indicator'], actuals_on_target['seen']))

        for _, row in production_df.iterrows():
            indicator = row['Indicator']
            forecast_tag = row[tag_col]
            was_seen = actuals_map.get(indicator, None)

            if target_date > today:
                outcome = "Pending"
            elif was_seen == 1:
                outcome = "Seen"
            elif was_seen == 0:
                outcome = "Not Seen"
            else:
                outcome = "Unknown"

            records.append({
                'Indicator': indicator,
                'Forecast Date': today,
                'Forecast Type': f'{h}-Day',
                'Confidence': forecast_tag,
                'Forecasted Check Date': target_date,
                'Outcome': outcome
            })

    df = pd.DataFrame(records)
    df['Forecast Date'] = pd.to_datetime(df['Forecast Date'])
    df['Forecasted Check Date'] = pd.to_datetime(df['Forecasted Check Date'])

    expected_columns = [
        'Indicator',
        'Forecast Date',
        'Forecast Type',
        'Confidence',
        'Forecasted Check Date',
        'Outcome'
    ]
    df = df[expected_columns]

    if log_csv_path and os.path.exists(log_csv_path):
        existing = pd.read_csv(log_csv_path, parse_dates=['Forecast Date', 'Forecasted Check Date'])
        df = pd.concat([existing, df], ignore_index=True).drop_duplicates()
        df = df[expected_columns]

    # Write to Excel with formatting
    if excel_output_path:
        df.to_excel(excel_output_path, index=False)

        wb = load_workbook(excel_output_path)
        ws = wb.active

        green_fill = PatternFill(start_color='CCFFCC', end_color='CCFFCC', fill_type='solid')  # Light green
        red_fill = PatternFill(start_color='FFCCCC', end_color='FFCCCC', fill_type='solid')    # Light red
        bold_font = Font(bold=True)

        # Apply conditional formatting
        for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
            row_data = {ws.cell(row=1, column=col_idx + 1).value: cell for col_idx, cell in enumerate(row)}
            confidence = row_data['Confidence'].value
            ftype = row_data['Forecast Type'].value
            outcome = row_data['Outcome'].value

            is_7day = ftype == '7-Day'
            is_highly_likely = 'Highly likely' in (confidence or '')
            is_low_confidence = 'Low confidence' in (confidence or '')
            is_seen = outcome == 'Seen'

            if is_7day and is_highly_likely and is_seen:
                for cell in row:
                    cell.fill = green_fill

            elif is_7day and is_low_confidence and not is_seen:
                for cell in row:
                    cell.fill = red_fill

            if is_seen:
                for cell in row:
                    cell.font = bold_font

        # Auto-fit column widths
        for col in ws.columns:
            max_length = 0
            col_letter = col[0].column_letter
            for cell in col:
                try:
                    value = str(cell.value) if cell.value is not None else ''
                    max_length = max(max_length, len(value))
                except:
                    continue
            adjusted_width = max_length + 2  # Padding
            ws.column_dimensions[col_letter].width = adjusted_width

        wb.save(excel_output_path)

    return df


In [22]:
import warnings

# Silence pandas groupby apply deprecation warning globally for this notebook
warnings.filterwarnings("ignore", category=DeprecationWarning, message="DataFrameGroupBy.apply operated on the grouping columns")

from datetime import datetime
import os

def main():
    today = datetime.today().date()

    prediction_path = r'C:\Users\jaskew\Documents\NOI Logs\Predictions'
    log_path = r'C:\Users\jaskew\Documents\NOI Logs'

    opdiv_production_outputs = {}
    opdiv_forecast_logs = {}
    production_output = None
    opdiv_df = None

    for opdiv_name, opdiv_df in opdiv_merged.items():
        opdiv_df = opdiv_df.copy()
        features_df = build_features(opdiv_df)
        output = get_model_outputs(features_df, opdiv_df)
        output = add_rule_and_ensemble(output)
        output = add_confidence_and_format(output)
        production_output = build_production_output(output)


        # Save production output
        opdiv_production_outputs[opdiv_name] = production_output

        # Define path for existing forecast log CSV
        opdiv_log_dir = os.path.join(log_path, opdiv_name)
        os.makedirs(opdiv_log_dir, exist_ok=True)
        log_csv_path = os.path.join(opdiv_log_dir, f"{opdiv_name}_forecast_log.csv")

        # Update the log (merge with previous if exists)
        forecast_log = update_long_forecast_log_with_formatting(production_output, opdiv_df, log_csv_path)
        opdiv_forecast_logs[opdiv_name] = forecast_log

        # Save updated forecast log
        #forecast_log.to_csv(log_csv_path, index=False)

        # Save today's prediction CSV
        opdiv_output_dir = os.path.join(prediction_path, opdiv_name)
        os.makedirs(opdiv_output_dir, exist_ok=True)
        output_csv_path = os.path.join(opdiv_output_dir, f"{opdiv_name}_output_{today.strftime('%Y%m%d')}.csv")
        #production_output.to_csv(output_csv_path, index=False)

    # Explicit unpacking of key OpDivs (optional for clarity)
    DHA_output = opdiv_production_outputs.get("DHA")
    CDC_output = opdiv_production_outputs.get("CDC")
    FDA_output = opdiv_production_outputs.get("FDA")
    NIH_output = opdiv_production_outputs.get("NIH")
    VA_output = opdiv_production_outputs.get("VA")
    HRSA_output = opdiv_production_outputs.get("HRSA")
    IHS_output = opdiv_production_outputs.get("IHS")
    OS_output = opdiv_production_outputs.get("OS")
    CMS_output = opdiv_production_outputs.get("CMS")
    HHS_output = opdiv_production_outputs.get("HHS")

    DHA_log = opdiv_forecast_logs.get("DHA")
    CDC_log = opdiv_forecast_logs.get("CDC")
    FDA_log = opdiv_forecast_logs.get("FDA")
    NIH_log = opdiv_forecast_logs.get("NIH")
    VA_log = opdiv_forecast_logs.get("VA")
    HRSA_log = opdiv_forecast_logs.get("HRSA")
    IHS_log = opdiv_forecast_logs.get("IHS")
    OS_log = opdiv_forecast_logs.get("OS")
    CMS_log = opdiv_forecast_logs.get("CMS")
    HHS_log = opdiv_forecast_logs.get("HHS")

    display(production_output.head(5))
    display(opdiv_df.head(5))

    return {
        "DHA_output": DHA_output, "CDC_output": CDC_output, "FDA_output": FDA_output,
        "NIH_output": NIH_output, "VA_output": VA_output, "HRSA_output": HRSA_output,
        "IHS_output": IHS_output, "OS_output": OS_output, "CMS_output": CMS_output,
        "HHS_output": HHS_output,
        "DHA_log": DHA_log, "CDC_log": CDC_log, "FDA_log": FDA_log,
        "NIH_log": NIH_log, "VA_log": VA_log, "HRSA_log": HRSA_log,
        "IHS_log": IHS_log, "OS_log": OS_log, "CMS_log": CMS_log,
        "HHS_log": HHS_log
    }


if __name__ == "__main__":
    main()

,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day
0,1-you.njalla.no,0,1.0,2.0,67.15%,7-Day: Possibly active,94.83%,14-Day: Possibly active,100.0%,30-Day: Highly likely
1,102.90.61.13,0,0.0,0.0,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.0%,30-Day: Low confidence
2,102.91.94.193,0,0.0,0.0,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.0%,30-Day: Low confidence
3,103.108.231.67,0,0.0,0.0,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.56%,30-Day: Low confidence
4,103.120.176.224,0,0.0,1.0,10.94%,7-Day: Low confidence,61.88%,14-Day: Possibly active,73.94%,30-Day: Possibly active


,API_UserName,date,indicator,OpDiv,observations,dayofweek,is_weekend,day,month,seen
0,74198686107399967946,2025-03-21,102.90.61.13,VA,39,4,False,21,3,1
1,74198686107399967946,2025-03-21,102.91.94.193,VA,30,4,False,21,3,1
2,74198686107399967946,2025-03-21,103.225.136.166,VA,12,4,False,21,3,1
3,74198686107399967946,2025-03-21,122.187.101.182,VA,32,4,False,21,3,1
4,74198686107399967946,2025-03-21,123.136.100.101,VA,14,4,False,21,3,1


In [24]:
import os
import pandas as pd
from datetime import datetime
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font
from datetime import date
today_date = date.today()

def parse_forecast_string(s):
    try:
        parts = [p.strip() for p in s.split('->')]
        if len(parts) == 3:
            forecast_tag = parts[0]
            forecasted_check_date = pd.to_datetime(parts[1])
            outcome = parts[2]

            # Normalize outcome label
            if outcome == "Missed":
                outcome = "Not Seen"

            if ':' in forecast_tag:
                forecast_type, confidence = [x.strip() for x in forecast_tag.split(':', 1)]
            else:
                forecast_type, confidence = None, forecast_tag

            return forecast_type, confidence, forecasted_check_date, outcome
    except:
        pass
    return None, None, None, None


def convert_and_format_logs(folder_path, excel_output_path):
    all_records = []

    for fname in os.listdir(folder_path):
        if not fname.lower().endswith('.csv'):
            continue

        file_path = os.path.join(folder_path, fname)
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"Skipping {fname} (read error): {e}")
            continue

        forecast_cols = [col for col in df.columns if col.startswith("Forecast Result: ")]
        if not forecast_cols:
            continue

        for col in forecast_cols:
            forecast_date_str = col.replace("Forecast Result: ", "").strip()
            try:
                forecast_date = pd.to_datetime(forecast_date_str)
            except:
                continue

            for _, row in df.iterrows():
                indicator = row.get('Indicator')
                if pd.isna(indicator):
                    continue

                val = row.get(col)
                if not isinstance(val, str):
                    continue

                segments = val.split(' | ')
                for seg in segments:
                    forecast_type, confidence, target_date, outcome = parse_forecast_string(seg)
                    if forecast_type and confidence and target_date and outcome:
                        all_records.append({
                            'Indicator': indicator,
                            'Forecast Date': forecast_date,
                            'Forecast Type': forecast_type,
                            'Confidence': confidence,
                            'Forecasted Check Date': target_date,
                            'Outcome': outcome
                        })

    df = pd.DataFrame(all_records)
    if df.empty:
        print("No valid records found.")
        return

    df['Forecast Date'] = pd.to_datetime(df['Forecast Date'])
    df['Forecasted Check Date'] = pd.to_datetime(df['Forecasted Check Date'])
    df = df.sort_values(['Indicator', 'Forecast Date', 'Forecast Type'])

    expected_columns = [
        'Indicator', 'Forecast Date', 'Forecast Type',
        'Confidence', 'Forecasted Check Date', 'Outcome'
    ]
    df = df[expected_columns]

    # 🔄 Handle output path as folder or file
    if os.path.isdir(excel_output_path):
        today_str = datetime.today().strftime("%Y-%m-%d")
        excel_output_path = os.path.join(excel_output_path, f"forecast_log.xlsx")
    elif not excel_output_path.lower().endswith(".xlsx"):
        excel_output_path += ".xlsx"

    # Save to Excel
    df.to_excel(excel_output_path, index=False)

    # --- Formatting ---
    wb = load_workbook(excel_output_path)
    ws = wb.active

    green_fill = PatternFill(start_color='CCFFCC', end_color='CCFFCC', fill_type='solid')  # Light green
    red_fill = PatternFill(start_color='FFCCCC', end_color='FFCCCC', fill_type='solid')   # Light red
    bold_font = Font(bold=True)

    # Define clear/default style
    default_fill = PatternFill(fill_type=None)
    default_font = Font(bold=False)

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
        row_data = {ws.cell(row=1, column=col_idx + 1).value: cell for col_idx, cell in enumerate(row)}
        confidence = row_data['Confidence'].value
        outcome = row_data['Outcome'].value
        ftype = row_data['Forecast Type'].value
        check_date = row_data['Forecasted Check Date'].value

        if not isinstance(check_date, (datetime, pd.Timestamp)):
            # Clear formatting if date is invalid
            for cell in row:
                cell.fill = default_fill
                cell.font = default_font
            continue

        if check_date.date() != today_date:
            # Not today's check date — clear any existing formatting
            for cell in row:
                cell.fill = default_fill
                cell.font = default_font
            continue

        # This is a forecast check due today — apply formatting
        is_seen = outcome == 'Seen'
        is_low_confidence = 'Low confidence' in (confidence or '')
        is_highly_likely = 'Highly likely' in (confidence or '')
        is_7day = ftype == '7-Day'

        apply_green = (
            (is_low_confidence and not is_seen) or
            (is_7day and is_highly_likely and is_seen)
        )

        apply_red = (
            (is_low_confidence and is_seen) or
            (is_7day and is_highly_likely and not is_seen)
        )

        if apply_green:
            for cell in row:
                cell.fill = green_fill

        elif apply_red:
            for cell in row:
                cell.fill = red_fill

        else:
            for cell in row:
                cell.fill = default_fill  # fallback if no condition matched

        if is_seen:
            for cell in row:
                cell.font = bold_font
        else:
            for cell in row:
                cell.font = default_font



    # Auto-fit column widths
    for col in ws.columns:
        max_length = 0
        col_letter = col[0].column_letter
        for cell in col:
            try:
                value = str(cell.value) if cell.value is not None else ''
                max_length = max(max_length, len(value))
            except:
                continue
        adjusted_width = max_length + 2
        ws.column_dimensions[col_letter].width = adjusted_width

    wb.save(excel_output_path)
    print(f"Logs saved to: {excel_output_path}")


In [27]:
def batch_convert_logs_in_subfolders(root_folder):
    for subdir, dirs, files in os.walk(root_folder):
        xlsx_files = [f for f in files if f.lower().endswith('.xlsx')]
        if not xlsx_files:
            continue  # skip folders with no XLSX files

        print(f"Processing folder: {subdir}")
        try:
            convert_and_format_logs(
                folder_path=subdir,
                excel_output_path=subdir  # Save Excel into same subfolder
            )
        except Exception as e:
            print(f"Error in folder {subdir}: {e}")

batch_convert_logs_in_subfolders(
    root_folder=r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs"
)


Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\CDC
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\CMS
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\DHA
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\FDA
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\HHS
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\HRSA
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\IHS
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\NIH
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\OS
No valid records found.
Processing folder: Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs\VA
No valid records found.


In [15]:
import os
from datetime import datetime, date
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font

def fix_formatting_in_excel(file_path):
    try:
        wb = load_workbook(file_path)
        ws = wb.active
    except Exception as e:
        print(f" Error loading file: {file_path} — {e}")
        return

    # Style definitions
    green_fill = PatternFill(start_color='CCFFCC', end_color='CCFFCC', fill_type='solid')
    red_fill = PatternFill(start_color='FFCCCC', end_color='FFCCCC', fill_type='solid')
    default_fill = PatternFill(fill_type=None)
    bold_font = Font(bold=True)
    default_font = Font(bold=False)

    today = date.today()

    # Map header to column
    header = {cell.value: idx for idx, cell in enumerate(ws[1])}

    required_cols = ['Confidence', 'Forecast Type', 'Outcome', 'Forecasted Check Date']
    if not all(col in header for col in required_cols):
        print(f" Skipping: required columns missing in {file_path}")
        return

    for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
        confidence = row[header['Confidence']].value
        ftype = row[header['Forecast Type']].value
        outcome = row[header['Outcome']].value
        check_date = row[header['Forecasted Check Date']].value

        # Default reset
        for cell in row:
            cell.fill = default_fill
            cell.font = default_font

        if not isinstance(check_date, datetime) or check_date.date() != today:
            continue  # Only format for today’s check date

        is_seen = outcome == 'Seen'
        is_low_confidence = 'Low confidence' in (confidence or '')
        is_highly_likely = 'Highly likely' in (confidence or '')
        is_7day = ftype == '7-Day'

        apply_green = (
            (is_low_confidence and not is_seen) or
            (is_7day and is_highly_likely and is_seen)
        )
        apply_red = (
            (is_low_confidence and is_seen) or
            (is_7day and is_highly_likely and not is_seen)
        )

        if apply_green:
            for cell in row:
                cell.fill = green_fill
        elif apply_red:
            for cell in row:
                cell.fill = red_fill

        if is_seen:
            for cell in row:
                cell.font = bold_font

    # Auto-fit column widths
    for col in ws.columns:
        max_len = max((len(str(cell.value)) for cell in col if cell.value), default=0)
        col_letter = col[0].column_letter
        ws.column_dimensions[col_letter].width = max_len + 2

    wb.save(file_path)
    print(f" Updated formatting: {file_path}")


def fix_formatting_in_all_excel_logs(root_folder):
    for subdir, _, files in os.walk(root_folder):
        for file in files:
            if file.lower().endswith(".xlsx"):
                full_path = os.path.join(subdir, file)
                fix_formatting_in_excel(full_path)

fix_formatting_in_all_excel_logs(r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Logs")


In [213]:
import os
import pandas as pd
from datetime import datetime

data_path_specific = r'Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions'

def load_all_csvs_from_folders(root_path):
    """
    Recursively loads all CSV files from all subfolders under root_path into a single DataFrame.
    Only includes files whose name ends with today's date in YYYYMMDD format (e.g., 20250616.csv).
    Excludes folders with titles 'automation scripts', 'Logs', or 'LogsBackup'.
    Adds a column 'Partner' with the partner name (e.g., 'CDC', 'NIH', etc) inferred from the folder name.
    Each CSV is expected to have the same structure.
    """
    exclude_folders = {'automation scripts', 'Logs', 'LogsBackup'}
    all_dfs = []
    today_str = datetime.today().strftime("%Y%m%d")
    for dirpath, dirnames, filenames in os.walk(root_path):
        # Exclude directories if any part of the path matches excluded names
        if any(ex in os.path.normpath(dirpath).split(os.sep) for ex in exclude_folders):
            continue
        partner = os.path.basename(dirpath)
        for fname in filenames:
            if fname.lower().endswith(f"{today_str}.csv"):
                fpath = os.path.join(dirpath, fname)
                try:
                    df = pd.read_csv(fpath)
                    df['Partner'] = partner
                    all_dfs.append(df)
                except Exception as e:
                    print(f"Skipping {fpath}: {e}")
    if all_dfs:
        return pd.concat(all_dfs, ignore_index=True)
    else:
        print("No CSV files found for today.")
        return pd.DataFrame()

specific_search = load_all_csvs_from_folders(data_path_specific)
    
display(specific_search)

,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,Partner
0,102.129.153.158,0,0.0,0.0,0.31%,7-Day: Low confidence,0.87%,14-Day: Low confidence,9.0%,30-Day: Low confidence,CDC
1,102.129.153.43,0,0.0,0.0,0.18%,7-Day: Low confidence,0.44%,14-Day: Low confidence,5.54%,30-Day: Low confidence,CDC
2,102.129.153.71,0,1.0,2.0,64.17%,7-Day: Possibly active,87.64%,14-Day: Possibly active,100.0%,30-Day: Highly likely,CDC
3,102.165.16.161,0,0.0,0.0,0.01%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.01%,30-Day: Low confidence,CDC
4,103.133.107.28,0,0.0,1.0,7.34%,7-Day: Low confidence,16.96%,14-Day: Low confidence,69.27%,30-Day: Possibly active,CDC
...,...,...,...,...,...,...,...,...,...,...,...
7403,www.3u.com,0,0.0,0.0,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.01%,30-Day: Low confidence,VA
7404,www.deepseek.com,1,6.0,29.0,100.0%,7-Day: Highly likely,100.0%,14-Day: Highly likely,100.0%,30-Day: Highly likely,VA
7405,www.deepseek.com.cdn.cloudflare.net,1,6.0,27.0,100.0%,7-Day: Highly likely,100.0%,14-Day: Highly likely,100.0%,30-Day: Highly likely,VA
7406,www.filemail.com,0,0.0,1.0,6.85%,7-Day: Low confidence,14.38%,14-Day: Low confidence,80.21%,30-Day: Possibly active,VA


In [216]:
search_indicators = [
    "102.129.215.40",
    "141.98.252.143",
    "144.76.136.153",
    "146.70.172.227",
    "169.150.227.205",
    "169.150.227.230",
    "185.248.85.20",
    "188.127.224.127",
    "188.127.224.199",
    "198.251.88.196",
    "198.54.131.36",
    "31.171.154.54",
    "76.8.60.64",
    "ln5.sync.com",
    "secure.eu.internxt.com",
    "secure.internxt.com",
    "ws.onehub.com",
    "www.filemail.com"
]

# Drop duplicates so each (Indicator, Partner) pair appears only once (keep the first occurrence)
result = specific_search[specific_search['Indicator'].isin(search_indicators)][
    ['Indicator', 'Probability: 7-Day', 'Probability: 14-Day', 'Probability: 30-Day', 'Partner']
]
result = result.drop_duplicates(subset=['Indicator', 'Partner'], keep='first')
display(result)

,Indicator,Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Partner
478,146.70.172.227,59.1%,65.75%,74.88%,CMS
618,169.150.227.205,29.92%,92.51%,100.0%,CMS
619,169.150.227.230,27.49%,54.69%,100.0%,CMS
730,185.248.85.20,99.65%,100.0%,100.0%,CMS
957,31.171.154.54,0.0%,0.0%,0.17%,CMS
1329,ln5.sync.com,0.01%,0.01%,0.28%,CMS
1355,ws.onehub.com,66.18%,91.96%,100.0%,CMS
1361,www.filemail.com,6.48%,12.14%,68.17%,CMS
1470,141.98.252.143,0.05%,0.31%,4.33%,DHA
1571,169.150.227.205,0.0%,0.01%,0.04%,DHA


In [215]:
specific_search[specific_search['Indicator'] == '185.248.85.20']

,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,Partner
730,185.248.85.20,0,2.0,6.0,99.65%,7-Day: Highly likely,100.0%,14-Day: Highly likely,100.0%,30-Day: Highly likely,CMS
1638,185.248.85.20,0,1.0,8.0,91.02%,7-Day: Possibly active,100.0%,14-Day: Highly likely,100.0%,30-Day: Highly likely,DHA
2423,185.248.85.20,0,2.0,3.0,82.93%,7-Day: Highly likely,100.0%,14-Day: Highly likely,100.0%,30-Day: Highly likely,FDA
3576,185.248.85.20,0,2.0,3.0,92.09%,7-Day: Highly likely,100.0%,14-Day: Highly likely,100.0%,30-Day: Highly likely,HRSA
4228,185.248.85.20,0,2.0,3.0,93.66%,7-Day: Highly likely,100.0%,14-Day: Highly likely,100.0%,30-Day: Highly likely,IHS


In [28]:
import os
import pandas as pd
from datetime import date, datetime
from openpyxl import load_workbook, Workbook
from openpyxl.styles import PatternFill, Font

def update_long_forecast_log_with_formatting(production_df, actuals_df, excel_output_path):
    """
    Update a long-format Excel forecast log by merging new forecast records derived from production_df and actuals_df,
    preserving existing data, and applying conditional formatting.

    Parameters:
    - production_df (pd.DataFrame): DataFrame containing forecast production data with columns including 'Indicator' and
      confidence tags like 'Confidence: 7-Day', 'Confidence: 14-Day', 'Confidence: 30-Day'.
    - actuals_df (pd.DataFrame): DataFrame containing actual indicator sightings with columns 'date', 'indicator', and 'seen'.
    - excel_output_path (str): File path to the Excel forecast log to read and write.

    Returns:
    - pd.DataFrame: The final merged DataFrame written to the Excel file.
    
    Notes:
    - The function normalizes dates, merges new and existing records on keys,
      updates outcomes based on today's date, and applies green/red fill and bold font based on confidence and outcome.
    - If the Excel file does not exist, it will be created.
    """

    try:
        # Normalize inputs
        actuals_df['date'] = pd.to_datetime(actuals_df['date']).dt.normalize()
        actuals_df['indicator'] = actuals_df['indicator'].astype(str).str.strip().str.lower()
        production_df['Indicator'] = production_df['Indicator'].astype(str).str.strip().str.lower()

        today = actuals_df['date'].max().normalize()
        today_date = today.date()
        records = []
        today_actuals = dict(zip(actuals_df['indicator'], actuals_df['seen']))

        # Generate new records for today's check date
        for h in [7, 14, 30]:
            tag_col = f'Confidence: {h}-Day'
            forecast_date = today - pd.Timedelta(days=h)
            target_date = today  # always today for new records
            for _, row in production_df.iterrows():
                indicator = row['Indicator']
                forecast_tag = row.get(tag_col, None)
                was_seen = today_actuals.get(indicator, 0)
                outcome = "Seen" if was_seen == 1 else "Not Seen"
                records.append({
                    'Indicator': indicator,
                    'Forecast Date': forecast_date,
                    'Forecast Type': f'{h}-Day',
                    'Confidence': forecast_tag,
                    'Forecasted Check Date': target_date,
                    'Outcome': outcome
                })

        df_new = pd.DataFrame(records)
        df_new['Forecast Date'] = pd.to_datetime(df_new['Forecast Date'])
        df_new['Forecasted Check Date'] = pd.to_datetime(df_new['Forecasted Check Date'])

        def normalize_confidence_label(confidence_str):
            if not isinstance(confidence_str, str):
                return confidence_str
            return confidence_str.split(':')[-1].strip()

        df_new['Confidence'] = df_new['Confidence'].apply(normalize_confidence_label)

        expected_columns = ['Indicator', 'Forecast Date', 'Forecast Type', 'Confidence', 'Forecasted Check Date', 'Outcome']
        df_new = df_new[expected_columns]

        # Load existing data if file exists
        if excel_output_path and os.path.exists(excel_output_path):
            print(f"[INFO] Loading existing forecast log from: {excel_output_path}")
            df_existing = pd.read_excel(excel_output_path, parse_dates=['Forecast Date', 'Forecasted Check Date'])
            df_existing['Forecast Date'] = pd.to_datetime(df_existing['Forecast Date']).dt.normalize()
            df_existing['Forecasted Check Date'] = pd.to_datetime(df_existing['Forecasted Check Date']).dt.normalize()
            df_existing = df_existing[expected_columns]

            # Merge new data with existing data on unique keys to update or append
            merge_cols = ['Indicator', 'Forecast Date', 'Forecast Type', 'Forecasted Check Date']
            df_combined = pd.merge(df_existing, df_new, how='outer', on=merge_cols, suffixes=('_old', ''))

            # For columns to update, prefer new if exists, else old
            for col in ['Confidence', 'Outcome']:
                df_combined[col] = df_combined[col].combine_first(df_combined[f'{col}_old'])
                df_combined.drop(columns=[f'{col}_old'], inplace=True)

            df_final = df_combined[expected_columns]
        else:
            print(f"[INFO] No existing file found, creating new forecast log at: {excel_output_path}")
            df_final = df_new

        # Update outcomes for all records based on check date
        def update_outcome(row):
            check_date = pd.to_datetime(row['Forecasted Check Date']).date()
            if check_date > today_date:
                return "Pending"
            was_seen = today_actuals.get(row['Indicator'], 0)
            return "Seen" if was_seen == 1 else "Not Seen"

        df_final['Outcome'] = df_final.apply(update_outcome, axis=1)

        # Save to Excel and apply formatting
        if os.path.exists(excel_output_path):
            wb = load_workbook(excel_output_path)
            ws = wb.active
            start_row = 2
        else:
            wb = Workbook()
            ws = wb.active
            ws.append(expected_columns)
            start_row = 2

        existing_rows = {}
        for row_idx in range(2, ws.max_row + 1):
            key = tuple(
                ws.cell(row=row_idx, column=col_idx).value
                for col_idx in range(1, 5)
            )
            existing_rows[key] = row_idx

        for i, row in df_final.iterrows():
            key = (
                row['Indicator'],
                row['Forecast Date'].to_pydatetime() if hasattr(row['Forecast Date'], 'to_pydatetime') else row['Forecast Date'],
                row['Forecast Type'],
                row['Forecasted Check Date'].to_pydatetime() if hasattr(row['Forecasted Check Date'], 'to_pydatetime') else row['Forecasted Check Date']
            )
            if key in existing_rows:
                row_idx = existing_rows[key]
                for col_idx, col_name in enumerate(expected_columns, start=1):
                    ws.cell(row=row_idx, column=col_idx).value = row[col_name]
            else:
                ws.append([row[col] for col in expected_columns])

        # Formatting styles
        green_fill = PatternFill(start_color='CCFFCC', end_color='CCFFCC', fill_type='solid')
        red_fill = PatternFill(start_color='FFCCCC', end_color='FFCCCC', fill_type='solid')
        bold_font = Font(bold=True)
        default_fill = PatternFill(fill_type=None)
        default_font = Font(bold=False)

        for row in ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column):
            row_data = {ws.cell(row=1, column=col_idx + 1).value: cell for col_idx, cell in enumerate(row)}
            confidence = row_data['Confidence'].value
            ftype = row_data['Forecast Type'].value
            outcome = row_data['Outcome'].value
            check_date = row_data['Forecasted Check Date'].value

            if not isinstance(check_date, (datetime, pd.Timestamp)) or check_date.date() != today_date:
                for cell in row:
                    cell.fill = default_fill
                    cell.font = default_font
                continue

            is_7day = ftype == '7-Day'
            is_highly_likely = 'Highly likely' in (confidence or '')
            is_low_confidence = 'Low confidence' in (confidence or '')
            is_seen = outcome == 'Seen'
            is_not_seen = outcome == 'Not Seen'

            apply_green = (is_low_confidence and is_not_seen) or (is_7day and is_highly_likely and is_seen)
            apply_red = (is_low_confidence and is_seen) or (is_7day and is_highly_likely and is_not_seen)

            if apply_green:
                for cell in row:
                    cell.fill = green_fill
            elif apply_red:
                for cell in row:
                    cell.fill = red_fill
            else:
                for cell in row:
                    cell.fill = default_fill

            if is_seen:
                for cell in row:
                    cell.font = bold_font
            else:
                for cell in row:
                    cell.font = default_font

        # Auto-fit columns
        for col in ws.columns:
            max_len = 0
            col_letter = col[0].column_letter
            for cell in col:
                try:
                    val = str(cell.value) if cell.value else ''
                    max_len = max(max_len, len(val))
                except Exception:
                    pass
            ws.column_dimensions[col_letter].width = max_len + 2

        wb.save(excel_output_path)
        print(f"[INFO] Saved updated forecast log to: {excel_output_path}")

        return df_final

    except Exception as e:
        print(f"[ERROR] Exception occurred in update_long_forecast_log_with_formatting: {e}")
        raise


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Load your uploaded CSV files
production_df = pd.read_csv(r'Documents/production.csv')
actuals_df = pd.read_csv(r'Documents/actual.csv')

# Convert dates in actuals_df to datetime and normalize
actuals_df['date'] = pd.to_datetime(actuals_df['date']).dt.normalize()

# To simulate a future check date scenario:
# 1. Find the max date in actuals_df and set that as "function today"
function_today = actuals_df['date'].max()

# 2. Add an actuals record with a date in the future relative to function_today
future_date = function_today + timedelta(days=1)
future_record = pd.DataFrame({
    'date': [future_date],
    'indicator': ['future_ip'],
    'seen': [0]
})

actuals_df = pd.concat([actuals_df, future_record], ignore_index=True)

# Also add corresponding production forecast for 'future_ip' with confidence columns
new_prod_record = pd.DataFrame({
    'Indicator': ['future_ip'],
    'Confidence: 7-Day': ['7-Day: Possibly active'],
    'Confidence: 14-Day': ['14-Day: Possibly active'],
    'Confidence: 30-Day': ['30-Day: Possibly active']
})

production_df = pd.concat([production_df, new_prod_record], ignore_index=True)

# Now call your function with these updated DataFrames
output_path = 'test_forecast_log_with_uploaded_data.xlsx'

df_result = update_long_forecast_log_with_formatting(production_df, actuals_df, output_path)

# Show the rows with Outcome == 'Pending' (future check dates)
pending_rows = df_result[df_result['Outcome'] == 'Pending']
print("Rows with Pending Outcome (future check dates):")
print(pending_rows)


[INFO] Loading existing forecast log from: test_forecast_log_with_future_check_dates.xlsx
[INFO] Saved updated forecast log to: test_forecast_log_with_future_check_dates.xlsx


,Indicator,Forecast Date,Forecast Type,Confidence,Forecasted Check Date,Outcome
0,ip1,2025-05-25,30-Day,Low confidence,2025-06-24,Seen
1,ip1,2025-06-10,14-Day,Possibly active,2025-06-24,Seen
2,ip1,2025-06-17,7-Day,Highly likely,2025-06-24,Seen
3,ip2,2025-05-25,30-Day,Highly likely,2025-06-24,Not Seen
4,ip2,2025-06-10,14-Day,Highly likely,2025-06-24,Not Seen
5,ip2,2025-06-17,7-Day,Possibly active,2025-06-24,Not Seen
6,ip3,2025-05-25,30-Day,Possibly active,2025-06-24,Seen
7,ip3,2025-06-10,14-Day,Low confidence,2025-06-24,Seen
8,ip3,2025-06-17,7-Day,Low confidence,2025-06-24,Seen



Rows with Pending Outcome (future check dates):


,Indicator,Forecast Date,Forecast Type,Confidence,Forecasted Check Date,Outcome
